In [23]:
"""
File Name: osw_final_stats.py
Author: Evan Anderson
School: Boston College & ESADE
Project Overview: Uses raw TrackMan data to find pitching team leaders in multiple statistics, including xWOBA. Assists with the end of
season review of player performance.
"""

'\nFile Name: osw_final_stats.py\nAuthor: Evan Anderson\nSchool: Boston College & ESADE\nProject Overview: Uses raw TrackMan data to find pitching team leaders in multiple statistics, including xWOBA. Assists with the end of\nseason review of player performance.\n'

In [24]:
#Imports necessary libraries.
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
import pandas as pd

#Loads the master CSV with every pitch of the Waves 2026 season (Excel utilized to create master CSV).
df_raw=pd.read_csv('Example.csv')
df_raw=df_raw[['Pitcher','PitcherId','PitcherTeam','PitchCall','KorBB','PlayResult','RelSpeed','SpinRate','ExitSpeed','Angle']]
df_raw=df_raw[~df_raw['Pitcher'].isin(['Plasschaert, Zach','Applefield, Kalen','Lippman, Hank','Kahn, Matthew'])]

#Filters and cleans the data sheet to only include pitches thrown by the Waves pitcher.
df=df_raw[df_raw['PitcherTeam']=='OCE_STA6'].copy()
df=df.dropna(subset=['Pitcher','PitcherId','PitchCall'])

FileNotFoundError: [Errno 2] No such file or directory: 'OSW_Stats_Updated.csv'

In [ ]:
#Creates a data set for pitches where the at_bat came to an end.
df_result=df[(df['KorBB'].isin(['Strikeout','Walk'])|df['PitchCall'].isin(['InPlay','HitByPitch']))].copy()

#Creates and cleans a data set for pitches where the ball was put in play.
df_bip=df_result[df_result['PitchCall']=='InPlay'].copy()
df_bip=df_bip.dropna(subset=['ExitSpeed','Angle'])

In [ ]:
#Assigns weights for each possible outcome of a ball in play.
woba_weights={'Single':0.88,'Double':1.25,'Triple':1.50,'HomeRun':2.03,'Out':0.00,'FieldersChoice':0.00,'Error':0.00,'Sacrifice':0.00}
df_bip['wOBA_val']=df_bip['PlayResult'].map(woba_weights).fillna(0)

In [25]:
#Assigns columns to be dependent and independent variables for the regression analysis, and runs said regression.
X=df_bip[['ExitSpeed','Angle']]
Y=df_bip['wOBA_val']

model=RandomForestRegressor(n_estimators=150,max_depth=5, min_samples_leaf=8,random_state=5)
df_bip['x_wOBA_val']=cross_val_predict(model,X,Y,cv=5)

In [6]:
#Makes sure a df_result xwOBA column doesn't exist and merges the ball in play expected data back into the overall end of at bat data set.
if 'x_wOBA_val' in df_result.columns:
    df_result=df_result.drop(columns=['x_wOBA_val'])
df_result=df_result.merge(df_bip[['x_wOBA_val']], left_index=True, right_index=True, how='left')

In [7]:
#Reassigns the weights to the different play results for the overall data set.
df_result['wOBA_val']=df_result['PlayResult'].map(woba_weights).fillna(0)

#Copies over the expected values.

#Fill in the wOBA values for the strikeouts, walks, and hit-by-pitches.
df_result.loc[df_result['KorBB']=='Walk','x_wOBA_val']=0.69
df_result.loc[df_result['KorBB']=='Walk','wOBA_val']=0.69

df_result.loc[df_result['PitchCall']=='HitByPitch','x_wOBA_val']=0.69
df_result.loc[df_result['PitchCall']=='HitByPitch','wOBA_val']=0.69


df_result.loc[df_result['KorBB']=='Strikeout','x_wOBA_val']=0.00
df_result.loc[df_result['KorBB']=='Strikeout','wOBA_val']=0.00

In [8]:
#Create binary numbers for strikeouts and walks for the additional statistical aggregations.
df_result['is_k']=(df_result['KorBB']=='Strikeout').astype(int)
df_result['is_bb']=(df_result['KorBB']=='Walk').astype(int)

In [9]:
#Creates important statistics that will assist in finding important pitching metrics.
pitch_stats=df_result.groupby('Pitcher').agg(total_pa=('Pitcher','count'),total_k=('is_k','sum'),total_bb=('is_bb','sum')).copy()

#Creates the statistics to calculate whiff rate.
df['is_swing']=df['PitchCall'].isin(['StrikeSwinging','FoulBallFieldable','FoulBallNotFieldable','InPlay']).astype(int)
df['is_whiff']=(df['PitchCall']=='StrikeSwinging').astype(int)
pitch_count=df.groupby('Pitcher').agg(total_swing=('is_swing','sum'),total_whiff=('is_whiff','sum'))

In [10]:
#Creates the final stats used for my project.
pitch_leaders=df_result.groupby('Pitcher').agg(pitcher_wOBA=('wOBA_val','mean'),pitcher_xwOBA=('x_wOBA_val','mean'),max_velo=('RelSpeed','max')
                                                  ,max_spin=('SpinRate','max'))
pitch_leaders['total_pa']=pitch_stats['total_pa']
pitch_leaders['k_rate']=(pitch_stats['total_k']/pitch_stats['total_pa'])*100
pitch_leaders['bb_rate']=(pitch_stats['total_bb']/pitch_stats['total_pa'])*100
pitch_leaders['whiff_rate']=(pitch_count['total_whiff']/pitch_count['total_swing'])*100

#Rounds the stats to standard decimal places.
pitch_leaders=pitch_leaders.round({'pitcher_wOBA':3,'pitcher_xwOBA':3,'max_velo':1,'max_spin':0,'k_rate':1,'bb_rate':1,'whiff_rate':1})
pitch_leaders['max_spin']=pitch_leaders['max_spin'].astype(int)

#Uses a final threshold of 30 plate appearances for pitchers to join the leaderboards.
pitch_leaders=pitch_leaders[pitch_leaders['total_pa']>30]
pitch_leaders.sort_values(by='pitcher_wOBA')

,pitcher_wOBA,pitcher_xwOBA,max_velo,max_spin,total_pa,k_rate,bb_rate,whiff_rate
Pitcher,,,,,,,,
"Hickman, Benton",0.198,0.216,92.4,2505,46,34.8,10.9,34.6
"Telfer, Trey",0.235,0.255,95.1,2836,68,32.4,17.6,38.5
"Eaton, Elliott",0.238,0.279,89.6,3223,77,26.0,11.7,21.2
"Waddle, Cooper",0.260,0.284,93.7,2857,48,43.8,22.9,41.7
"Maher, Cooper",0.274,0.310,95.4,2467,81,25.9,12.3,26.3
"Causey, Dylan",0.276,0.301,91.6,2608,93,20.4,10.8,23.5
"Faulcon, Kayden",0.283,0.317,89.2,2867,172,18.6,8.7,21.2
"Yoak, Bryant",0.287,0.298,97.7,2874,44,43.2,22.7,42.9
"McCarthy, Dylan",0.300,0.241,93.8,2574,51,31.4,11.8,33.3


In [11]:
#pitcher wOBA leaders
print("wOBA leaders:")
display(pitch_leaders.sort_values(by='pitcher_wOBA',ascending=True)[['pitcher_wOBA']].head())
print()

#pitcher velocity leaders
print("Maximum velocity leaders:")
display(pitch_leaders.sort_values(by='max_velo',ascending=False)[['max_velo']].head())
print()

#pitcher spin rate leaders
print("Maximum spin rate leaders:")
display(pitch_leaders.sort_values(by='max_spin',ascending=False)[['max_spin']].head())
print()

#pitcher strikeout rate leaders
print("Strikeout rate leaders:")
display(pitch_leaders.sort_values(by='k_rate',ascending=False)[['k_rate']].head())
print()

#pitcher walk rate leaders
print("Walk rate leaders:")
display(pitch_leaders.sort_values(by='bb_rate',ascending=True)[['bb_rate']].head())
print()

#pitcher whiff rate leaders
print("Whiff rate leaders:")
display(pitch_leaders.sort_values(by='whiff_rate',ascending=False)[['whiff_rate']].head())
print()

#pitcher xwOBA leaders
print("Expected wOBA leaders:")
display(pitch_leaders.sort_values(by='pitcher_xwOBA',ascending=True)[['pitcher_xwOBA']].head())

wOBA leaders:


,pitcher_wOBA
Pitcher,
"Hickman, Benton",0.198
"Telfer, Trey",0.235
"Eaton, Elliott",0.238
"Waddle, Cooper",0.260
"Maher, Cooper",0.274



Maximum velocity leaders:


,max_velo
Pitcher,
"Yoak, Bryant",97.7
"Flannery, Boston",95.8
"Maher, Cooper",95.4
"Telfer, Trey",95.1
"Barton, Hudson",94.2



Maximum spin rate leaders:


...

,bb_rate
Pitcher,
"Billings, Luke",8.0
"Flannery, Boston",8.5
"Faulcon, Kayden",8.7
"Confreda, Brennan",9.5
"Causey, Dylan",10.8



Whiff rate leaders:


,whiff_rate
Pitcher,
"Yoak, Bryant",42.9
"Waddle, Cooper",41.7
"Telfer, Trey",38.5
"Jacobs, Andrew",37.4
"Hickman, Benton",34.6



Expected wOBA leaders:


,pitcher_xwOBA
Pitcher,
"Hickman, Benton",0.216
"McCarthy, Dylan",0.241
"Telfer, Trey",0.255
"Eaton, Elliott",0.279
"Waddle, Cooper",0.284
